# CMO-MPC (Koh 2020) Suction Test Evaluation Runner
**Project:** Physiological Coordinated Control of Biventricular Artificial Hearts (BiVAD)  
**Task:** Re-evaluate classical MPC on the physics-correct acute volume-evacuation Suction Scenario (Scenario 9).

---
### Parallel Execution Strategy on Kaggle (4 vCPUs):
- **Session Type:** Notebook (CPU-Only, No GPU required)
- **Strategy A (Fastest: ~55-60 min total):**
  - **Notebook 1:** Run 4 seeds simultaneously across 4 vCPUs (`SEEDS = '42,123,456,789'`).
  - **Notebook 2:** Run seed 1024 on a 2nd tab (`SEEDS = '1024'`).
  - *Both notebooks finish in ~1 hour!*
- **Strategy B (Hands-Off: ~110 min total):**
  - Run all 5 seeds in a single notebook (`SEEDS = '42,123,456,789,1024'`).
  - Runs 4 seeds in parallel, and as soon as one completes, the 5th seed runs automatically.

### Block 0: Configuration
Choose which seeds to run on this notebook instance.

In [ ]:
# Set seeds to run on this notebook:
# Options:
#   '42,123,456,789'   -> Runs 4 seeds in parallel on 4 cores (~60 min)
#   '1024'             -> Runs seed 1024 (~60 min)
#   '42,123,456,789,1024' -> Runs all 5 seeds on 1 notebook (~110 min)
SEEDS = "42,123,456,789"
WORKERS = 4

print(f"Configured MPC Suction Runner: Seeds={SEEDS} | Workers={WORKERS}")

### Block 1: Environment Setup & File Extraction

In [ ]:
# 1. Install dependencies
!pip install gymnasium scipy pandas tabulate

# 2. Locate and extract files
import os, glob, shutil, zipfile, multiprocessing

zip_candidates = (
    glob.glob('/kaggle/input/**/kaggle_training_test.zip', recursive=True) +
    glob.glob('/kaggle/input/**/kaggle_bivad_*.zip', recursive=True) +
    glob.glob('kaggle_training_test.zip')
)
bivad_env_matches = glob.glob('/kaggle/input/**/bivad_env.py', recursive=True)

if zip_candidates:
    target_zip = zip_candidates[0]
    print(f"Unzipping {target_zip}...")
    with zipfile.ZipFile(target_zip, 'r') as z:
        z.extractall('.')
elif bivad_env_matches:
    src = os.path.dirname(bivad_env_matches[0])
    print(f"Copying files from {src}...")
    for item in os.listdir(src):
        s = os.path.join(src, item)
        d = os.path.join('.', item)
        if os.path.isdir(s):
            if os.path.exists(d): shutil.rmtree(d)
            shutil.copytree(s, d)
        else:
            shutil.copy2(s, d)
else:
    print("Assuming files are in working directory.")

# 3. Verify
for f in ['simulator', 'controllers', 'bivad_env.py', 'run_mpc_suction.py']:
    print(f"{f:22s}: {'[FOUND]' if os.path.exists(f) else '[MISSING]'}")
print(f"\nDetected CPU Cores: {multiprocessing.cpu_count()}")

### Block 2: Run Parallel MPC Suction Test Evaluation
Prints step progress updates every 200 steps so you can monitor execution.

In [ ]:
!python run_mpc_suction.py --seeds {SEEDS} --workers {WORKERS} --output_dir ./mpc_output

### Block 3: Package Results for Download
Zips the resulting CSV into `mpc_suction_results.zip` for instant download.

In [ ]:
!zip -r mpc_suction_results.zip mpc_output/
print("\n>>> [READY TO DOWNLOAD] 'mpc_suction_results.zip' is ready in the Kaggle Output panel!")